# 👑 System One — Milestone 2 & 3 (Colab A100): Two-Stage MoS Residual Scar Experts (`Qwen3-Embedding-8B` 8-Depth)
Este notebook executa o treinamento fim-a-fim em **2 Estágios** dos nossos **Especialistas de Cicatriz Residuais MoS** (`rank=16`, zero LoRAs de terceiros) acoplados à extração simultânea de **8 Profundidades (`[4, 8, 12, 16, 20, 24, 30, 36]`)** e **Janelas Deslizantes Sem Truncamento (`max_len=512, stride=384`)**:
1. **Estágio 1:** Adaptação geral no corpus limpo desduplicado (`pool_v4.parquet` + `passages.csv` proxy-matched, sem sobreposição com `train.csv`).
2. **Estágio 2:** Calibração fina no registro exato do host (`train.csv`, 5-Fold `StratifiedGroupKFold` por `prompt_id`), exportando os 5 estados `.safetensors` e o backbone NF4 MoS diretamente para o Kaggle.

In [ ]:
!pip install -q -U bitsandbytes accelerate safetensors transformers kaggle pyarrow scikit-learn
import torch
print('GPU:', torch.cuda.get_device_name(0), '| BF16:', torch.cuda.is_bf16_supported())


### Etapa 1: Autenticação Kaggle & Download dos Corpora Oficiais e Estendidos Limpos
Faça upload do seu `kaggle.json` (ou defina no Colab Secrets) para baixar `train.csv`, `pool_v4.parquet` e `passages.csv` e publicar automaticamente os artefatos ao final.

In [ ]:
import os, pathlib
from google.colab import files
kaggle_dir = pathlib.Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
if not (kaggle_dir / 'kaggle.json').exists():
    uploaded = files.upload()
    for k, v in uploaded.items():
        (kaggle_dir / 'kaggle.json').write_bytes(v)
os.chmod(kaggle_dir / 'kaggle.json', 0o600)

!mkdir -p /content/ntl_data
!kaggle competitions download -c name-that-llm -f train.csv -p /content/ntl_data -q && unzip -o -q /content/ntl_data/train.csv.zip -d /content/ntl_data || true
!kaggle datasets download -d nishantdahal/ntl-handwriting-blend -f pool_v4.parquet -p /content/ntl_data --unzip -q
!kaggle datasets download -d prvsiyan/prvsiyans-extended-name-that-llm-dataset -f passages.csv -p /content/ntl_data --unzip -q
!ls -lh /content/ntl_data


### Etapa 2: Treinamento 2-Estágios dos Especialistas de Cicatriz MoS + 8-Depth Readout + `SCache O(1)`
Grava `train_two_stage_mos_scar_experts_colab.py` e executa o pipeline completo em BF16/NF4 no A100 (~45–55 min).

In [ ]:
from pathlib import Path
Path('/content/train_two_stage_mos_scar_experts_colab.py').write_text('#!/usr/bin/env python3\n"""\n👑 SYSTEM ONE — MILESTONE 2 & 3 (COLAB A100):\nTWO-STAGE PROPRIETARY MoS RESIDUAL SCAR EXPERTS + 8-DEPTH MULTI-SLICE ENGINE\n============================================================================\nAlvo de Leaderboard Público: >= 0.740 - 0.755+ (Top 10 em `name-that-llm`).\n100% Arquitetura Proprietária System One (`AGENTS.md` & `SKILL.md`):\n- ZERO adaptadores LoRA comunitários / de terceiros.\n- Especialistas de Cicatriz Residuais MoS (`dW_scar = scale * B @ A`, rank=16) injetados\n  diretamente em `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`, exportados\n  no formato `.scar_A` / `.scar_B` nativamente suportado pelo `extract_mos_nf4_streaming`!\n- Extração simultânea de 8 Profundidades (`[4, 8, 12, 16, 20, 24, 30, 36]` em `Qwen/Qwen3-Embedding-8B`)\n  com pesos aprendidos por camada + Janelas Deslizantes Sem Truncamento (`max_len=512, stride=384`).\n- Treinamento Fim-a-Fim em Mini-Batches Agrupados por Prompt: retropropaga simultaneamente\n  pelo `Plain Head`, pelo `Delta-Origin Prompt-Centered Twin Head (z - mu_prompt)` e pela\n  memória associativa `SCache O(1)`, ensinando as camadas internas a anular o assunto do prompt!\n\nEstágio 1 (Pré-Adaptação Forense em Larga Escala):\n  `pool_v4.parquet` + `passages.csv` limpo (desduplicado contra `train.csv` e sem `kimi-k3` legado).\nEstágio 2 (Calibração Fina de Registro do Host Kaggle):\n  5-Fold `StratifiedGroupKFold` por `prompt_id` sobre `train.csv` (2.699 linhas oficiais),\n  salvando 5 estados de cicatriz (`fold0..fold4_scar_experts.safetensors`) + `oof_preds.npy`\n  + backbone NF4 MoS pronto para upload direto ao Kaggle!\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport math\nimport os\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Dict, List, Sequence, Tuple\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom safetensors.torch import save_file\nfrom transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig\n\nLABELS = [\n    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",\n    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",\n    "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "claude-fable-5",\n    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3",\n]\nL2I = {l: i for i, l in enumerate(LABELS)}\nK = len(LABELS)\n\nCAPTURE_LAYERS_QWEN3_8B = [4, 8, 12, 16, 20, 24, 30, 36]\nSCAR_TARGET_MODULES = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")\n\n\n# ==============================================================================\n# 1. Camada Proprietária MoS Residual Scar Expert (Compatível com Streaming NF4)\n# ==============================================================================\nclass MoSResidualScarLinear(nn.Module):\n    """\n    Envolve uma projeção base quantizada NF4 congelada com um Especialista de Cicatriz\n    Residual de posto baixo (`y = base(x) + (alpha / r) * (drop(x) @ A.T @ B.T)`).\n    Serializa `prefix.scar_A` (r, in_features) e `prefix.scar_B` (out_features, r) em FP16\n    para fusão in-place `w_out.addmm_(sB, sA, alpha=scar_scale)` no motor de streaming MoS!\n    """\n\n    def __init__(self, base_linear: nn.Module, rank: int = 16, alpha: float = 32.0, dropout: float = 0.05):\n        super().__init__()\n        self.base = base_linear\n        for p in self.base.parameters():\n            p.requires_grad_(False)\n\n        in_features = int(base_linear.in_features)\n        out_features = int(base_linear.out_features)\n        self.rank = rank\n        self.scale = float(alpha) / float(rank)\n        self.drop = nn.Dropout(dropout)\n\n        dev = next(base_linear.parameters()).device\n        self.scar_A = nn.Parameter(torch.empty((rank, in_features), device=dev, dtype=torch.bfloat16))\n        self.scar_B = nn.Parameter(torch.zeros((out_features, rank), device=dev, dtype=torch.bfloat16))\n        nn.init.kaiming_uniform_(self.scar_A, a=math.sqrt(5))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        base_out = self.base(x)\n        x_bf = self.drop(x.to(self.scar_A.dtype))\n        delta = F.linear(F.linear(x_bf, self.scar_A), self.scar_B) * self.scale\n        return base_out + delta.to(base_out.dtype)\n\n\ndef inject_mos_scar_experts(backbone: nn.Module, rank: int = 16, alpha: float = 32.0, dropout: float = 0.05) -> Dict[str, MoSResidualScarLinear]:\n    """Congela 100% do backbone base (incluindo embeddings/layernorms) e injeta `MoSResidualScarLinear`."""\n    for p in backbone.parameters():\n        p.requires_grad_(False)\n    injected: Dict[str, MoSResidualScarLinear] = {}\n    for mod_name, mod in list(backbone.named_modules()):\n        for child_name, child in list(mod.named_children()):\n            if child_name in SCAR_TARGET_MODULES and hasattr(child, "in_features") and hasattr(child, "out_features"):\n                full_prefix = f"{mod_name}.{child_name}" if mod_name else child_name\n                scar_mod = MoSResidualScarLinear(child, rank=rank, alpha=alpha, dropout=dropout)\n                setattr(mod, child_name, scar_mod)\n                injected[full_prefix] = scar_mod\n    return injected\n\n\n# ==============================================================================\n# 2. Cabeça Multi-Depth de 8 Camadas + SCache O(1) + Dual Plain/Prompt-Twin Readout\n# ==============================================================================\nclass MultiDepthMoSSCacheClassifier(nn.Module):\n    """\n    Combina 8 fatias de profundidade (`[4,8,12,16,20,24,30,36]`) extraídas num único forward:\n      1. Normalização RMS unitária por fatia + LayerNorm aprendida por profundidade\n      2. Ponderação softmax aprendida entre profundidades + projeção residual `8*D -> proj_dim`\n      3. Modulação associativa `SCache O(1)` intra-prompt (`32x32` covariância de cicatriz)\n      4. Cabeças gêmeas simultâneas: `Plain Head` + `Prompt-Centered Twin Head (z - mu_prompt)`\n    """\n\n    def __init__(\n        self,\n        backbone: nn.Module,\n        d_model: int,\n        capture_layers_1b: Sequence[int] = CAPTURE_LAYERS_QWEN3_8B,\n        proj_dim: int = 512,\n        scache_dim: int = 32,\n        num_classes: int = 18,\n    ):\n        super().__init__()\n        self.backbone = backbone\n        self.d_model = d_model\n        self.capture_layers_1b = list(capture_layers_1b)\n        self.n_depths = len(self.capture_layers_1b)\n\n        self.depth_norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(self.n_depths)])\n        self.depth_logits = nn.Parameter(torch.zeros(self.n_depths))\n        self.proj = nn.Sequential(\n            nn.Linear(self.n_depths * d_model, proj_dim),\n            nn.LayerNorm(proj_dim),\n            nn.SiLU(),\n            nn.Dropout(0.20),\n        )\n        self.scache_key = nn.Linear(proj_dim, scache_dim, bias=False)\n        self.scache_scale = nn.Parameter(torch.tensor(0.15))\n\n        self.plain_norm = nn.LayerNorm(proj_dim)\n        self.plain_head = nn.Linear(proj_dim, num_classes)\n\n        self.twin_norm = nn.LayerNorm(proj_dim)\n        self.twin_head = nn.Linear(proj_dim, num_classes)\n\n    def encode_passages(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n        out = self.backbone(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            output_hidden_states=True,\n            use_cache=False,\n        )\n        # hidden_states[0] = embeddings; hidden_states[l_1b] = saída da camada 1-based `l_1b`\n        hs = out.hidden_states\n        m = attention_mask.unsqueeze(-1).float()\n        denom = m.sum(dim=1).clamp(min=1.0)\n\n        pooled_slices = []\n        d_weights = F.softmax(self.depth_logits, dim=0) * self.n_depths\n        n_tot_layers = int(self.backbone.config.num_hidden_layers)\n        eps = float(getattr(self.backbone.config, "rms_norm_eps", 1e-6))\n        for idx, l_1b in enumerate(self.capture_layers_1b):\n            h_l = hs[l_1b].float()\n            if l_1b == n_tot_layers:\n                h_use = h_l\n            else:\n                h_use = h_l * torch.rsqrt(h_l.pow(2).mean(dim=-1, keepdim=True) + eps)\n            p = (h_use * m).sum(dim=1) / denom\n            p_unit = p * torch.rsqrt(p.pow(2).mean(dim=-1, keepdim=True) + 1e-6)\n            p_norm = self.depth_norms[idx](p_unit) * d_weights[idx]\n            pooled_slices.append(p_norm)\n\n        z_cat = torch.cat(pooled_slices, dim=-1)\n        return self.proj(z_cat)\n\n    def apply_scache_and_heads(\n        self, z: torch.Tensor, prompt_ids: torch.Tensor | None = None\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        """\n        Aplica ressonância `SCache O(1)` e centralização por grupo quando `prompt_ids` possui irmãos.\n        """\n        z_sc = z\n        z_centered = torch.zeros_like(z)\n\n        if prompt_ids is not None:\n            s_proj = self.scache_key(z)\n            z_sc_list = z.clone()\n            for g in torch.unique(prompt_ids):\n                mask = prompt_ids == g\n                cnt = int(mask.sum().item())\n                if cnt > 1:\n                    sg = s_proj[mask]\n                    dz = sg - sg.mean(dim=0, keepdim=True)\n                    w_scar = (dz.transpose(0, 1) @ dz) / float(cnt)\n                    res = (dz @ w_scar.transpose(0, 1)) @ self.scache_key.weight\n                    z_sc_list[mask] = z[mask] + self.scache_scale * res\n            z_sc = z_sc_list\n            for g in torch.unique(prompt_ids):\n                mask = prompt_ids == g\n                if int(mask.sum().item()) > 1:\n                    z_centered[mask] = z_sc[mask] - z_sc[mask].mean(dim=0, keepdim=True)\n\n        logits_plain = self.plain_head(self.plain_norm(z_sc))\n        logits_twin = self.twin_head(self.twin_norm(z_centered))\n        return logits_plain, logits_twin\n\n\n# ==============================================================================\n# 3. Construtor de Corpus Desduplicado 2-Estágios & Amostrador Agrupado por Prompt\n# ==============================================================================\ndef load_clean_stage1_corpus(train_csv: Path, pool_v4_path: Path, passages_csv: Path):\n    import pandas as pd\n    df_tr = pd.read_csv(train_csv)\n    tr_texts = set(df_tr["text"].fillna("").astype(str).str.strip())\n    tr_pids = set(df_tr["prompt_id"].astype(str))\n\n    frames = []\n    if pool_v4_path.exists():\n        pv4 = pd.read_parquet(pool_v4_path)\n        pv4 = pv4[pv4["model"].isin(L2I)].copy()\n        pv4["text_clean"] = pv4["text"].fillna("").astype(str).str.strip()\n        pv4 = pv4[(~pv4["text_clean"].isin(tr_texts)) & (~pv4["prompt_id"].astype(str).isin(tr_pids))]\n        frames.append(\n            pd.DataFrame({\n                "prompt_id": "pv4_" + pv4["prompt_id"].astype(str),\n                "text": pv4["text"].astype(str),\n                "y": pv4["model"].map(L2I).values,\n                "src_pid": "",\n            })\n        )\n\n    if passages_csv.exists():\n        pa = pd.read_csv(passages_csv)\n        pa = pa[\n            (pa["passes_funnel"] == True)\n            & (pa["is_refusal"] == False)\n            & (pa["is_empty"] == False)\n            & (pa["split"] != "pilot_range")\n            & (~((pa["split"] == "new") & (pa["model"] == "kimi-k3")))\n            & (pa["model"].isin(L2I))\n        ].copy()\n        pa["text_clean"] = pa["text"].fillna("").astype(str).str.strip()\n        # Evita duplicar linhas de `pa` que já estão em `pv4`\n        existing = set(frames[0]["text"].str.strip()) if frames else set()\n        pa = pa[(~pa["text_clean"].isin(tr_texts)) & (~pa["text_clean"].isin(existing))]\n        frames.append(\n            pd.DataFrame({\n                "prompt_id": "pa_" + pa["prompt_id"].astype(str),\n                "text": pa["text"].astype(str),\n                "y": pa["model"].map(L2I).values,\n                "src_pid": pa["source_prompt_id"].fillna("").astype(str),\n            })\n        )\n\n    full_ext = pd.concat(frames, ignore_index=True)\n    print(f"✓ Stage-1 Clean Deduplicated Corpus: {len(full_ext)} passages across {full_ext[\'prompt_id\'].nunique()} prompts")\n    return full_ext\n\n\ndef build_prompt_grouped_batches(df, batch_size: int = 16, seed: int = 42) -> List[np.ndarray]:\n    """Agrupa índices do mesmo `prompt_id` no mesmo mini-batch para ativar `z - mu_prompt` fim-a-fim."""\n    rng = np.random.default_rng(seed)\n    by_prompt: Dict[str, List[int]] = defaultdict(list)\n    for idx, pid in enumerate(df["prompt_id"].astype(str).values):\n        by_prompt[pid].append(idx)\n\n    pids = list(by_prompt.keys())\n    rng.shuffle(pids)\n    batches: List[np.ndarray] = []\n    cur: List[int] = []\n    for pid in pids:\n        rows = by_prompt[pid].copy()\n        rng.shuffle(rows)\n        for r in rows:\n            cur.append(r)\n            if len(cur) == batch_size:\n                batches.append(np.asarray(cur, dtype=np.int64))\n                cur = []\n    if len(cur) >= 2:\n        batches.append(np.asarray(cur, dtype=np.int64))\n    return batches\n\n\n# ==============================================================================\n# 4. Exportador de Pesos `.safetensors` Compatível com `extract_mos_nf4_streaming`\n# ==============================================================================\ndef export_fold_safetensors(\n    model: MultiDepthMoSSCacheClassifier,\n    injected: Dict[str, MoSResidualScarLinear],\n    out_path: Path,\n):\n    tensors: Dict[str, torch.Tensor] = {}\n    for prefix, scar_mod in injected.items():\n        # Salva scar_A já multiplicado por `scale` para fusão direta `w_out.addmm_(sB, sA, alpha=1.0)`!\n        tensors[f"{prefix}.scar_A"] = (scar_mod.scar_A.detach().float() * scar_mod.scale).to(torch.float16).cpu()\n        tensors[f"{prefix}.scar_B"] = scar_mod.scar_B.detach().to(torch.float16).cpu()\n\n    # Salva parâmetros da cabeça Multi-Depth + SCache\n    for k, v in model.state_dict().items():\n        if not k.startswith("backbone."):\n            tensors[f"head.{k}"] = v.detach().float().cpu()\n\n    save_file(tensors, str(out_path))\n\n\n@torch.no_grad()\ndef evaluate_covering_windows(\n    model: MultiDepthMoSSCacheClassifier,\n    tokenizer,\n    texts: List[str],\n    prompt_groups: np.ndarray,\n    max_len: int = 512,\n    stride: int = 384,\n    batch_size: int = 32,\n    device: str = "cuda:0",\n) -> np.ndarray:\n    """Inferência completa sem truncamento via janelas deslizantes ponderadas por tokens."""\n    model.eval()\n    enc_all = tokenizer(texts, truncation=False, padding=False)["input_ids"]\n\n    win_ids, doc_idx, win_wts = [], [], []\n    for i, seq in enumerate(enc_all):\n        L = len(seq)\n        if L <= max_len:\n            win_ids.append(seq if L > 0 else [tokenizer.eos_token_id])\n            doc_idx.append(i)\n            win_wts.append(float(max(L, 1)))\n        else:\n            starts = list(range(0, L - max_len + 1, stride))\n            if starts[-1] + max_len < L:\n                starts.append(L - max_len)\n            for st in starts:\n                ch = seq[st : st + max_len]\n                win_ids.append(ch)\n                doc_idx.append(i)\n                win_wts.append(float(len(ch)))\n\n    order = np.argsort([len(s) for s in win_ids], kind="stable")\n    proj_dim = model.plain_head.in_features\n    Z_win = np.zeros((len(win_ids), proj_dim), dtype=np.float32)\n\n    for b_s in range(0, len(win_ids), batch_size):\n        b_i = order[b_s : b_s + batch_size]\n        tok_b = tokenizer.pad({"input_ids": [win_ids[j] for j in b_i]}, padding=True, return_tensors="pt").to(device)\n        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):\n            z_b = model.encode_passages(tok_b["input_ids"], tok_b["attention_mask"])\n        Z_win[b_i] = z_b.float().cpu().numpy()\n\n    Z_doc = np.zeros((len(texts), proj_dim), dtype=np.float32)\n    w_sum = np.zeros((len(texts), 1), dtype=np.float32)\n    doc_idx_arr = np.asarray(doc_idx, dtype=np.int64)\n    win_wts_arr = np.asarray(win_wts, dtype=np.float32)\n    np.add.at(Z_doc, doc_idx_arr, Z_win * win_wts_arr[:, None])\n    np.add.at(w_sum[:, 0], doc_idx_arr, win_wts_arr)\n    Z_doc /= np.maximum(w_sum, 1e-6)\n\n    # Aplica SCache + Plain + Twin sobre os documentos agregados\n    z_t = torch.from_numpy(Z_doc).to(device)\n    _, u_inv = np.unique(prompt_groups, return_inverse=True)\n    g_t = torch.from_numpy(u_inv).to(device)\n    logits_p, logits_c = model.apply_scache_and_heads(z_t, g_t)\n    probs = 0.45 * F.softmax(logits_p, dim=-1) + 0.55 * F.softmax(logits_c, dim=-1)\n    return probs.cpu().numpy()\n\n\n# ==============================================================================\n# 5. Entrypoint CLI para Execução Turn-Key no Google Colab A100\n# ==============================================================================\ndef main():\n    import pandas as pd\n    from sklearn.metrics import balanced_accuracy_score, log_loss\n    from sklearn.model_selection import StratifiedGroupKFold\n    parser = argparse.ArgumentParser(description="System One Two-Stage MoS Residual Scar Experts (Colab A100)")\n    parser.add_argument("--model-id", default="Qwen/Qwen3-Embedding-8B")\n    parser.add_argument("--train-csv", required=True, help="Caminho para train.csv oficial (2.699 linhas)")\n    parser.add_argument("--pool-v4", required=True, help="Caminho para pool_v4.parquet")\n    parser.add_argument("--passages-csv", required=True, help="Caminho para passages.csv")\n    parser.add_argument("--out-dir", default="./titan4_qwen3_emb_8b_mos_artifacts")\n    parser.add_argument("--stage1-max-rows", type=int, default=36000, help="Amostragem balanceada Stage 1 (~2000/LLM)")\n    parser.add_argument("--stage1-epochs", type=int, default=2)\n    parser.add_argument("--stage2-epochs", type=int, default=4)\n    parser.add_argument("--batch-size", type=int, default=16)\n    parser.add_argument("--rank", type=int, default=16)\n    args = parser.parse_args()\n\n    out_dir = Path(args.out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    device = "cuda:0" if torch.cuda.is_available() else "cpu"\n    print(f"🚀 Launching Two-Stage MoS Scar-Expert Training on {device} | Model: {args.model_id}")\n\n    df_train = pd.read_csv(args.train_csv)\n    df_train["text"] = df_train["text"].fillna("").astype(str)\n    df_train["y"] = df_train["model"].map(L2I).values\n    prompts_train = df_train["prompt_id"].astype(str).values\n\n    df_ext = load_clean_stage1_corpus(Path(args.train_csv), Path(args.pool_v4), Path(args.passages_csv))\n    if len(df_ext) > args.stage1_max_rows:\n        per_cls = args.stage1_max_rows // K\n        df_ext = (\n            df_ext.groupby("y", group_keys=False)\n            .apply(lambda g: g.sample(n=min(len(g), per_cls), random_state=42))\n            .reset_index(drop=True)\n        )\n        print(f"✓ Balanced Stage-1 Subset: {len(df_ext)} rows ({per_cls}/class)")\n\n    tokenizer = AutoTokenizer.from_pretrained(args.model_id, padding_side="left")\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n\n    bnb_cfg = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type="nf4",\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_compute_dtype=torch.bfloat16,\n    )\n    print("⬇ Carregando e quantizando backbone NF4 MoS base...")\n    base_backbone = AutoModel.from_pretrained(args.model_id, quantization_config=bnb_cfg, device_map={"": 0})\n    base_backbone.gradient_checkpointing_enable()\n\n    # Salva uma vez o backbone NF4 quantizado para upload ao Kaggle (compatível com extract_mos_nf4_streaming!)\n    bb_save_dir = out_dir / "backbone_nf4_mos"\n    if not (bb_save_dir / "config.json").exists():\n        print(f"💾 Exportando backbone NF4 MoS para {bb_save_dir}...")\n        base_backbone.save_pretrained(bb_save_dir, safe_serialization=True, max_shard_size="4GB")\n        tokenizer.save_pretrained(bb_save_dir)\n\n    d_model = int(base_backbone.config.hidden_size)\n    injected = inject_mos_scar_experts(base_backbone, rank=args.rank, alpha=2 * args.rank, dropout=0.05)\n    model = MultiDepthMoSSCacheClassifier(base_backbone, d_model=d_model).to(device)\n    # Mantém cabeças em FP32 para estabilidade numérica sob autocast BF16\n    for name, p in model.named_parameters():\n        if not name.startswith("backbone."):\n            p.data = p.data.float()\n\n    trainable_params = [p for p in model.parameters() if p.requires_grad]\n    print(f"✓ Injected {len(injected)} MoS Residual Scar projections | Trainable params: {sum(p.numel() for p in trainable_params):,}")\n\n    def run_epoch(df_ep: pd.DataFrame, opt: torch.optim.Optimizer, lr_sched, seed: int) -> float:\n        model.train()\n        batches = build_prompt_grouped_batches(df_ep, batch_size=args.batch_size, seed=seed)\n        tot_loss, n_steps = 0.0, 0\n        pids_all = df_ep["prompt_id"].astype(str).values\n        texts_all = df_ep["text"].tolist()\n        y_all = df_ep["y"].values\n\n        for b_idx in batches:\n            b_texts = [texts_all[i] for i in b_idx]\n            _, u_inv = np.unique(pids_all[b_idx], return_inverse=True)\n            tok = tokenizer(b_texts, truncation=True, max_length=512, padding=True, return_tensors="pt").to(device)\n            yt = torch.from_numpy(y_all[b_idx]).long().to(device)\n            gt = torch.from_numpy(u_inv).long().to(device)\n\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):\n                z = model.encode_passages(tok["input_ids"], tok["attention_mask"])\n                lp, lc = model.apply_scache_and_heads(z.float(), gt)\n                loss = 0.45 * F.cross_entropy(lp, yt, label_smoothing=0.05) + 0.55 * F.cross_entropy(lc, yt, label_smoothing=0.05)\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)\n            opt.step()\n            if lr_sched is not None:\n                lr_sched.step()\n            tot_loss += float(loss.item())\n            n_steps += 1\n        return tot_loss / max(n_steps, 1)\n\n    # ==========================================================================\n    # ESTÁGIO 1: Adaptação Geral no Open Pool Limpo (Salva Checkpoint Base Stage-1)\n    # ==========================================================================\n    print("\\n=== [STAGE 1] Pre-Adapting MoS Residual Scar Experts on Clean Open Pool ===")\n    # Filtra `train_regen` do Estágio 1 global para preservar 100% de independência para todos os 5 folds!\n    df_ext_stage1 = df_ext[df_ext["src_pid"] == ""].reset_index(drop=True)\n    total_s1_steps = max(len(build_prompt_grouped_batches(df_ext_stage1, args.batch_size)) * args.stage1_epochs, 1)\n    opt_s1 = torch.optim.AdamW(trainable_params, lr=2e-4, weight_decay=0.02)\n    sched_s1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s1, T_max=total_s1_steps, eta_min=2e-5)\n\n    for ep in range(args.stage1_epochs):\n        t0 = time.time()\n        l_ep = run_epoch(df_ext_stage1, opt_s1, sched_s1, seed=100 + ep)\n        print(f"  Stage-1 Epoch {ep+1}/{args.stage1_epochs} | Loss={l_ep:.4f} | Time={time.time()-t0:.1f}s", flush=True)\n\n    # Guarda snapshot em RAM dos pesos treináveis pós-Stage-1 para ramificar cada um dos 5 Folds Stage-2!\n    stage1_snapshot = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if any(t in k for t in ("scar_A", "scar_B")) or not k.startswith("backbone.")}\n\n    # ==========================================================================\n    # ESTÁGIO 2: Calibração Fina de Registro do Host em 5-Fold StratifiedGroupKFold\n    # ==========================================================================\n    print("\\n=== [STAGE 2] 5-Fold Host-Register Calibration on Official `train.csv` ===")\n    sgkf = StratifiedGroupKFold(n_splits=5)\n    oof_probs = np.zeros((len(df_train), K), dtype=np.float32)\n\n    for fold_i, (tr_idx, va_idx) in enumerate(sgkf.split(df_train, df_train["y"].values, groups=prompts_train)):\n        tf0 = time.time()\n        # Restaura estado Stage-1 limpo\n        cur_sd = model.state_dict()\n        for k, v in stage1_snapshot.items():\n            cur_sd[k].copy_(v.to(cur_sd[k].device))\n\n        df_fold_tr = df_train.iloc[tr_idx][["prompt_id", "text", "y"]].reset_index(drop=True)\n        total_s2_steps = max(len(build_prompt_grouped_batches(df_fold_tr, args.batch_size)) * args.stage2_epochs, 1)\n        opt_s2 = torch.optim.AdamW(trainable_params, lr=8e-5, weight_decay=0.03)\n        sched_s2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s2, T_max=total_s2_steps, eta_min=5e-6)\n\n        for ep in range(args.stage2_epochs):\n            run_epoch(df_fold_tr, opt_s2, sched_s2, seed=500 + fold_i * 10 + ep)\n\n        va_texts = df_train.iloc[va_idx]["text"].tolist()\n        va_groups = prompts_train[va_idx]\n        p_va = evaluate_covering_windows(model, tokenizer, va_texts, va_groups, device=device)\n        oof_probs[va_idx] = p_va\n\n        fold_acc = balanced_accuracy_score(df_train["y"].values[va_idx], p_va.argmax(axis=1))\n        fold_path = out_dir / f"titan4_qwen3_emb_8b_fold{fold_i}_scars.safetensors"\n        export_fold_safetensors(model, injected, fold_path)\n        print(f"  ✓ Fold {fold_i+1}/5 | Host Val BalAcc = {fold_acc*100:.2f}% | Saved {fold_path.name} ({time.time()-tf0:.1f}s)", flush=True)\n        gc.collect()\n        torch.cuda.empty_cache()\n\n    oof_bal = balanced_accuracy_score(df_train["y"].values, oof_probs.argmax(axis=1))\n    oof_ll = log_loss(df_train["y"].values, np.clip(oof_probs, 1e-9, 1.0))\n    np.save(out_dir / "oof_titan4_qwen3_emb_8b_8depth.npy", oof_probs)\n    meta = {\n        "model_id": args.model_id,\n        "capture_layers_1b": CAPTURE_LAYERS_QWEN3_8B,\n        "rank": args.rank,\n        "oof_balanced_accuracy": float(oof_bal),\n        "oof_log_loss": float(oof_ll),\n        "labels": LABELS,\n    }\n    (out_dir / "meta_titan4_qwen3_emb_8b.json").write_text(json.dumps(meta, indent=2))\n    print("=" * 80)\n    print(f"🏆 TITAN 4 (Qwen3-Embedding-8B 8-Depth MoS Scar Experts) 5-FOLD OOF BALANCED ACC: {oof_bal*100:.2f}% | LL: {oof_ll:.4f}")\n    print("=" * 80)\n\n\nif __name__ == "__main__":\n    main()\n')
!python3 /content/train_two_stage_mos_scar_experts_colab.py \
  --model-id Qwen/Qwen3-Embedding-8B \
  --train-csv /content/ntl_data/train.csv \
  --pool-v4 /content/ntl_data/pool_v4.parquet \
  --passages-csv /content/ntl_data/passages.csv \
  --out-dir /content/titan4_qwen3_emb_8b_mos_artifacts \
  --stage1-max-rows 36000 --stage1-epochs 2 --stage2-epochs 4 --batch-size 16 --rank 16


### Etapa 3: Publicação Automática dos Datasets Privados no Kaggle (`Backbone NF4` + `5-Fold Scar Experts`)
Publica `thetod/name-that-llm-titan4-qwen3emb-mos-backbone` e `thetod/name-that-llm-titan4-qwen3emb-mos-scars` prontos para acoplamento imediato no kernel `thetod/mixture-of-scars-70b-mos-scache-name-that-llm`.

In [ ]:
import json, shutil
from pathlib import Path

art = Path('/content/titan4_qwen3_emb_8b_mos_artifacts')
bb_dir = art / 'backbone_nf4_mos'
(bb_dir / 'dataset-metadata.json').write_text(json.dumps({
    'title': 'Name That LLM Titan 4 Qwen3 Emb MoS Backbone',
    'id': 'thetod/name-that-llm-titan4-qwen3emb-mos-backbone',
    'licenses': [{'name': 'CC0-1.0'}]
}, indent=2))

scars_dir = Path('/content/titan4_scars_upload')
scars_dir.mkdir(exist_ok=True)
for f in art.glob('*.safetensors'): shutil.copy2(f, scars_dir / f.name)
for f in art.glob('*.json'): shutil.copy2(f, scars_dir / f.name)
for f in art.glob('*.npy'): shutil.copy2(f, scars_dir / f.name)
(scars_dir / 'dataset-metadata.json').write_text(json.dumps({
    'title': 'Name That LLM Titan 4 Qwen3 Emb MoS Scars',
    'id': 'thetod/name-that-llm-titan4-qwen3emb-mos-scars',
    'licenses': [{'name': 'CC0-1.0'}]
}, indent=2))

!kaggle datasets create -p /content/titan4_qwen3_emb_8b_mos_artifacts/backbone_nf4_mos --dir-mode zip || kaggle datasets version -p /content/titan4_qwen3_emb_8b_mos_artifacts/backbone_nf4_mos -m 'Update Titan 4 NF4 MoS Backbone' --dir-mode zip
!kaggle datasets create -p /content/titan4_scars_upload --dir-mode zip || kaggle datasets version -p /content/titan4_scars_upload -m 'Update Titan 4 5-Fold MoS Scar Experts' --dir-mode zip


### Conclusão
Após concluir o upload acima, basta anexar `thetod/name-that-llm-titan4-qwen3emb-mos-backbone` e `thetod/name-that-llm-titan4-qwen3emb-mos-scars` ao nosso kernel System One Apex — o motor `extract_mos_nf4_streaming` já detecta e funde automaticamente as 8 profundidades e os tensores `.scar_A`/`.scar_B` em `< 2.5 GiB` de VRAM!